In [6]:
# pip install soundfile
from dataclasses import dataclass
from pathlib import Path
import soundfile as sf

In [7]:
SUPPORTED_EXTS = {".wav", ".mp3"}
MAX_SIZE_MB = 100


@dataclass
class AudioInput:
    path: Path
    format: str
    size_mb: float
    duration_sec: float
    sample_rate: int
    channels: int


def load_audio(file_path: str) -> AudioInput:
    """Validate an audio file and return its basic metadata."""
    path = Path(file_path)

    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

    ext = path.suffix.lower()
    if ext not in SUPPORTED_EXTS:
        raise ValueError(f"Unsupported format '{ext}'. Use one of {SUPPORTED_EXTS}")

    size_mb = path.stat().st_size / (1024 * 1024)
    if size_mb == 0:
        raise ValueError("File is empty")
    if size_mb > MAX_SIZE_MB:
        raise ValueError(f"File too large: {size_mb:.1f} MB (max {MAX_SIZE_MB} MB)")

    try:
        info = sf.info(str(path))  # reads header only, fast
    except Exception as e:
        raise ValueError(f"Could not read audio file (corrupt or unsupported): {e}")

    return AudioInput(
        path=path,
        format=ext.lstrip("."),
        size_mb=round(size_mb, 2),
        duration_sec=round(info.duration, 2),
        sample_rate=info.samplerate,
        channels=info.channels,
    )


# Usage
audio = load_audio("sample_audio.wav")
print(audio)

AudioInput(path=PosixPath('sample_audio.wav'), format='wav', size_mb=1.0, duration_sec=5.94, sample_rate=44100, channels=2)
